<a href="https://colab.research.google.com/github/js880514/ROT7_calculation_tool/blob/main/ROT7_calculation_tool.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from ipywidgets import interact, FloatSlider, IntSlider, Layout, Dropdown

def calculate_and_plot_v16(ab_dist_input, cd_dist_input, R_prime, s_offset, w_offset, mode):
    # ==========================================
    # 1. 依據英文模式 (Mode) 決定實際經緯長度
    # ==========================================
    if mode == '1 into 1':
        ab_dist = ab_dist_input
        cd_dist = cd_dist_input
    elif mode == '1 into 2(weft)':
        ab_dist = ab_dist_input
        cd_dist = cd_dist_input * 2
    elif mode == '1 into 2(warp)':
        ab_dist = ab_dist_input * 2
        cd_dist = cd_dist_input
    elif mode == '1 into 4':
        ab_dist = ab_dist_input * 2
        cd_dist = cd_dist_input * 2

    r_deg = 90 - R_prime
    s = s_offset
    w = w_offset

    # ==========================================
    # 2. 基礎角度與三角函數計算 (轉換為弧度)
    # ==========================================
    r = np.radians(r_deg)
    r_90 = np.radians(r_deg + 90)

    tan_r = np.tan(r)
    tan_r90 = np.tan(r_90)

    inv_cos_r = np.sqrt(tan_r**2 + 1)
    inv_cos_r90 = np.sqrt(tan_r90**2 + 1)

    # ==========================================
    # 3. 計算原始參數與各線的 y 截距
    # ==========================================
    n = (ab_dist * inv_cos_r) / np.abs(tan_r)
    b_C = -(10 + n) * tan_r90
    m = b_C + cd_dist * inv_cos_r90

    b_A = -10 * tan_r
    b_B = -(10 + n) * tan_r
    b_D = m

    # 計算平行線 G 與 H 的 y 截距
    b_G = (b_A + b_B) / 2
    b_H = (b_C + b_D) / 2

    # A', B', C', D' 的 y 截距
    b_A_prime = b_A + (s / 2) * inv_cos_r
    b_B_prime = b_B - (s / 2) * inv_cos_r
    b_C_prime = b_C - (s / 2) * inv_cos_r90
    b_D_prime = b_D + (s / 2) * inv_cos_r90

    # ==========================================
    # 4. 解交點與定義水平線
    # ==========================================
    def intersect(k1, b1, k2, b2):
        x = (b2 - b1) / (k1 - k2)
        y = k1 * x + b1
        return np.array([x, y])

    # 原始 ABCD 頂點
    P_AD = intersect(tan_r, b_A, tan_r90, b_D)
    P_BD = intersect(tan_r, b_B, tan_r90, b_D)
    P_BC = intersect(tan_r, b_B, tan_r90, b_C)
    P_AC = intersect(tan_r, b_A, tan_r90, b_C)

    # 特殊幾何交點
    P_AH = intersect(tan_r, b_A, tan_r90, b_H)
    P_BH = intersect(tan_r, b_B, tan_r90, b_H)
    P_CG = intersect(tan_r, b_G, tan_r90, b_C)
    P_DG = intersect(tan_r, b_G, tan_r90, b_D)

    # A'B'C'D' 頂點
    P_A_prime_D_prime = intersect(tan_r, b_A_prime, tan_r90, b_D_prime)
    P_B_prime_D_prime = intersect(tan_r, b_B_prime, tan_r90, b_D_prime)
    P_B_prime_C_prime = intersect(tan_r, b_B_prime, tan_r90, b_C_prime)
    P_A_prime_C_prime = intersect(tan_r, b_A_prime, tan_r90, b_C_prime)

    # 水平線 Y 座標定義
    y_E_prime = P_A_prime_D_prime[1]
    y_F_prime = P_B_prime_C_prime[1]

    y_E_double_prime = y_E_prime + (w / 2)
    y_F_double_prime = y_F_prime - (w / 2)

    b_A_double_prime = b_A_prime + (w / 2) * inv_cos_r
    b_B_double_prime = b_B_prime - (w / 2) * inv_cos_r

    # 裁切特殊交點
    x_B_prime_E_prime = (y_E_prime - b_B_prime) / tan_r
    P_B_prime_E_prime = np.array([x_B_prime_E_prime, y_E_prime])

    x_A_prime_F_prime = (y_F_prime - b_A_prime) / tan_r
    P_A_prime_F_prime = np.array([x_A_prime_F_prime, y_F_prime])

    # A''B''E''F'' 的四個頂點
    P_A_double_prime_E_double_prime = np.array([(y_E_double_prime - b_A_double_prime) / tan_r, y_E_double_prime])
    P_B_double_prime_E_double_prime = np.array([(y_E_double_prime - b_B_double_prime) / tan_r, y_E_double_prime])
    P_B_double_prime_F_double_prime = np.array([(y_F_double_prime - b_B_double_prime) / tan_r, y_F_double_prime])
    P_A_double_prime_F_double_prime = np.array([(y_F_double_prime - b_A_double_prime) / tan_r, y_F_double_prime])

    # ==========================================
    # 5. 距離與面積計算 (鞋帶公式)
    # ==========================================
    def polygon_area(pts):
        x = pts[:, 0]
        y = pts[:, 1]
        return 0.5 * np.abs(np.dot(x, np.roll(y, 1)) - np.dot(y, np.roll(x, 1)))

    dist_glass_fabric_lon = np.linalg.norm(P_A_double_prime_F_double_prime - P_B_double_prime_F_double_prime)
    dist_glass_fabric_lat = np.abs(y_E_double_prime - y_F_double_prime)

    pts_ABCD = np.array([P_AD, P_BD, P_BC, P_AC])
    pts_A_prime_B_prime_C_prime_D_prime = np.array([P_A_prime_D_prime, P_B_prime_D_prime, P_B_prime_C_prime, P_A_prime_C_prime])
    pts_A_prime_B_prime_E_prime_F_prime_frame = np.array([P_A_prime_D_prime, P_B_prime_E_prime, P_B_prime_C_prime, P_A_prime_F_prime])
    pts_A_double_prime_B_double_prime_E_double_prime_F_double_prime = np.array([
        P_A_double_prime_E_double_prime, P_B_double_prime_E_double_prime,
        P_B_double_prime_F_double_prime, P_A_double_prime_F_double_prime
    ])

    area_ABCD = polygon_area(pts_ABCD)
    area_A_prime_B_prime_C_prime_D_prime = polygon_area(pts_A_prime_B_prime_C_prime_D_prime)
    area_A_prime_B_prime_E_prime_F_prime = polygon_area(pts_A_prime_B_prime_E_prime_F_prime_frame)
    area_total_glass_fabric = polygon_area(pts_A_double_prime_B_double_prime_E_double_prime_F_double_prime)

    area_tri_sum = polygon_area(np.array([P_A_prime_D_prime, P_B_prime_D_prime, P_B_prime_E_prime])) + \
                   polygon_area(np.array([P_A_prime_C_prime, P_B_prime_C_prime, P_A_prime_F_prime]))

    area_trim_reserve = np.abs(area_total_glass_fabric - area_A_prime_B_prime_E_prime_F_prime)
    area_bs_shrink_reserve = np.abs(area_A_prime_B_prime_C_prime_D_prime - area_ABCD)

    pct_trim = (area_trim_reserve / area_total_glass_fabric) * 100
    pct_scrap = (area_tri_sum / area_total_glass_fabric) * 100
    pct_bs_reserve = (area_bs_shrink_reserve / area_total_glass_fabric) * 100
    pct_utilization = (area_ABCD / area_total_glass_fabric) * 100

    # ==========================================
    # 6. 依指令分層繪製圖形
    # ==========================================
    fig, ax = plt.subplots(figsize=(15, 10))

    all_pts = np.vstack([pts_ABCD, pts_A_prime_B_prime_C_prime_D_prime, pts_A_double_prime_B_double_prime_E_double_prime_F_double_prime])
    x_min, y_min = np.min(all_pts, axis=0) - 15
    x_max, y_max = np.max(all_pts, axis=0) + 15

    plot_width = x_max - x_min
    ax.set_xlim(x_min, x_max + plot_width * 0.45)
    ax.set_ylim(y_min, y_max)

    # 【第 0 層】：輕微灰點網格
    ax.grid(True, linestyle=':', color='#CCCCCC', alpha=0.6, zorder=0)

    # 【第 1 層】：背景全區帶狀塗色 (鮮黃色)
    ax.axhspan(y_F_double_prime, y_E_double_prime, facecolor='#FFFF33', alpha=1.0, zorder=1)

    # 【第 2 層】：A''B''E''F'' 內 (淺藍色)
    poly_A2B2E2F2 = patches.Polygon(pts_A_double_prime_B_double_prime_E_double_prime_F_double_prime,
                                    facecolor='#CCFFFF', edgecolor='none', alpha=1.0, zorder=2)
    ax.add_patch(poly_A2B2E2F2)

    # 【第 3 層】：補角三角形 (深一點的藍色 royalblue)
    poly_BDE_prime = patches.Polygon(np.array([P_A_prime_D_prime, P_B_prime_D_prime, P_B_prime_E_prime]),
                                     facecolor='royalblue', edgecolor='none', alpha=1.0, zorder=3)
    poly_ACF_prime = patches.Polygon(np.array([P_A_prime_C_prime, P_B_prime_C_prime, P_A_prime_F_prime]),
                                     facecolor='royalblue', edgecolor='none', alpha=1.0, zorder=3)
    ax.add_patch(poly_BDE_prime)
    ax.add_patch(poly_ACF_prime)

    # 【第 4 層】：A'B'C'D' 內 (再深的藍色 dodgerblue)
    poly_A1B1C1D1 = patches.Polygon(pts_A_prime_B_prime_C_prime_D_prime,
                                    facecolor='dodgerblue', edgecolor='none', alpha=1.0, zorder=4)
    ax.add_patch(poly_A1B1C1D1)

    # 【第 5 層】：ABCD 核心區域填滿純白，壓上紅色十字交叉網格 (+)
    poly_ABCD_white = patches.Polygon(pts_ABCD, facecolor='#FFFFFF', edgecolor='none', alpha=1.0, zorder=5)
    poly_ABCD_hatch = patches.Polygon(pts_ABCD, facecolor='none', edgecolor='#FF3333', hatch='+', lw=0, zorder=5)
    ax.add_patch(poly_ABCD_white)
    ax.add_patch(poly_ABCD_hatch)

    # 【第 6 層】：頂層精細線框
    ax.axhline(y=y_E_double_prime, color='green', linestyle='--', linewidth=2, zorder=6)
    ax.axhline(y=y_F_double_prime, color='purple', linestyle='--', linewidth=2, zorder=6)

    x_track = np.array([x_min, x_max + plot_width * 0.45])
    y_A2_track = tan_r * x_track + b_A_double_prime
    y_B2_track = tan_r * x_track + b_B_double_prime
    ax.plot(x_track, y_A2_track, color='#888888', linestyle=':', linewidth=0.8, zorder=6)
    ax.plot(x_track, y_B2_track, color='#888888', linestyle=':', linewidth=0.8, zorder=6)

    # ─── 【依據英文選擇模式進行動態虛線繪製】 ───
    if mode in ['1 into 2(weft)', '1 into 4']:
        # 繪製 H 線段 (點 AH 到點 BH)
        ax.plot([P_AH[0], P_BH[0]], [P_AH[1], P_BH[1]], color='#666666', linestyle='--', linewidth=1.5, zorder=7)

    if mode in ['1 into 2(warp)', '1 into 4']:
        # 繪製 G 線段 (點 CG 到點 DG)
        ax.plot([P_CG[0], P_DG[0]], [P_CG[1], P_DG[1]], color='#666666', linestyle='--', linewidth=1.5, zorder=7)

    # 畫布約束
    ax.set_aspect('equal', adjustable='box')
    ax.set_title(f"Process Layout Analysis - Mode: {mode}", fontsize=14, pad=15)
    ax.set_xlabel("X Axis")
    ax.set_ylabel("Y Axis")

    # 右側純英文數據面板 (全面英文化)
    info_text = (
        f"[ Length Data ]\n"
        f"  Layout Mode: {mode}\n"
        f"  Rotation Angle (R'): {R_prime}°\n"
        f"  BS Warp: {ab_dist:.2f}\n"
        f"  BS Weft: {cd_dist:.2f}\n"
        f"  GF Warp: {dist_glass_fabric_lon:.2f}\n"
        f"  GF Weft: {dist_glass_fabric_lat:.2f}\n\n"
        f"[ Ratio Analysis ]\n"
        f"  GF Allowance Ratio: {pct_trim:.2f}%\n"
        f"  BS Cutting Scrap Ratio: {pct_scrap:.2f}%\n"
        f"  BS Allowance Ratio: {pct_bs_reserve:.2f}%\n"
        f"  Product Utilization Rate: {pct_utilization:.2f}%"
    )

    ax.text(0.72, 0.95, info_text, transform=ax.transAxes, fontsize=12,
            verticalalignment='top', horizontalalignment='left',
            fontname='DejaVu Sans',
            bbox=dict(boxstyle='round,pad=0.6', facecolor='#F8F9FA', alpha=0.95, edgecolor='#CED4DA', linewidth=1.5))

    plt.tight_layout()
    plt.show()

    # ==========================================
    # 7. 下方終端機數據文字輸出
    # ==========================================
    print("="*70)
    print(f"【製程排版數據計算結果 - 模式: {mode}】")
    print("-"*70)
    print(f"1. 旋轉角度 (Rotation Angle)               = {R_prime}°")
    print(f"2. BS 經長 (BS Warp)                      = {ab_dist:.6f}")
    print(f"3. BS 緯長 (BS Weft)                      = {cd_dist:.6f}")
    print(f"4. 玻布經長 (GF Warp)                      = {dist_glass_fabric_lon:.6f}")
    print(f"5. 玻布緯長 (GF Weft)                      = {dist_glass_fabric_lat:.6f}")
    print(f"6. 單位玻布總面積                         = {area_total_glass_fabric:.6f}")
    print(f"7. 玻布預留修邊面積                       = {area_trim_reserve:.6f}")
    print(f"   -> GF Allowance Ratio                  = {pct_trim:.2f}%")
    print(f"8. 裁切報廢面積 (補角加總面積)             = {area_tri_sum:.6f}")
    print(f"   -> BS Cutting Scrap Ratio              = {pct_scrap:.2f}%")
    print(f"9. BS分切/收縮預留面積                    = {area_bs_shrink_reserve:.6f}")
    print(f"   -> BS Allowance Ratio                  = {pct_bs_reserve:.2f}%")
    print(f"10. 實際產品尺寸 (核心面積)                = {area_ABCD:.6f}")
    print(f"   -> Product Utilization Rate            = {pct_utilization:.2f}%")
    print("="*70)

# ==========================================
# 8. 建立互動式拉桿與國際化下拉選單
# ==========================================
interact(
    calculate_and_plot_v16,
    ab_dist_input=FloatSlider(value=21.45, min=1.0, max=100.0, step=0.01, description='BS經長:', style={'description_width': 'initial'}, layout=Layout(width='60%')),
    cd_dist_input=FloatSlider(value=24.45, min=1.0, max=100.0, step=0.01, description='BS緯長:', style={'description_width': 'initial'}, layout=Layout(width='60%')),
    R_prime=IntSlider(value=7, min=1, max=89, step=1, description="旋轉角度:", style={'description_width': 'initial'}, layout=Layout(width='60%')),
    s_offset=FloatSlider(value=1.0, min=0.0, max=30.0, step=0.01, description='BS分切/收縮預留長度:', style={'description_width': 'initial'}, layout=Layout(width='60%')),
    w_offset=FloatSlider(value=2.0, min=0.0, max=30.0, step=0.01, description='玻布修邊預留長度:', style={'description_width': 'initial'}, layout=Layout(width='60%')),
    mode=Dropdown(options=['1 into 1', '1 into 2(weft)', '1 into 2(warp)', '1 into 4'], value='1 into 1', description='Layout Mode:', style={'description_width': 'initial'}, layout=Layout(width='60%'))
);

interactive(children=(FloatSlider(value=21.45, description='BS經長:', layout=Layout(width='60%'), min=1.0, step=…